# Treinando classificadores de texto com Aprendizado Profundo

### Instalando pacotes

In [1]:
!pip install transformers accelerate sentencepiece torchinfo datasets

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from torchinfo import summary
import os

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 24.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.4/261.4 kB 30.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 47.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 493.7/493.7 kB 51.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.1/311.1 kB 38.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 63.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 56.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 17.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 18.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.0/295.0 kB 37.0 MB/s eta 0:00:00


### Abrindo a base

Abra a base utilizada no Colab da aula 28 (é um CSV!), chame-o de "data" e renomeie a coluna "ementa" para "text". Conte quantos projetos existem em cada categoria e, em seguida, salve a base como "data.csv" aqui no Colab.

In [2]:
data = pd.read_csv('https://github.com/mateuspestana/LabExtracaoAnalise_ECMI/raw/main/bases/iser.csv')
data.rename(columns={'ementa': 'text'}, inplace=True)
data.to_csv('data.csv')

In [3]:
label_count = data['label'].value_counts()
label_count

Educação                                                   384
Direitos Da Mulher                                         288
Crimes E Segurança Pública                                 270
Causa Animal                                               194
Direitos Da Criança E Adolescente                          160
Direitos Humanos                                           146
Regulação Das Mídias                                       107
Calendários Oficiais, Patrimonialização E Homenagens       102
Raça E Racismo                                              90
Direitos Dos Povos Indígenas E Comunidades Tradicionais     84
Direitos Sexuais E Reprodutivos                             81
Ameaça Ao Estado Democrático De Direito                     71
Família                                                     58
Políticas De Drogas                                         57
Religião                                                    55
Imigração                                              

In [4]:
data.value_counts('label').shape

(16,)

### Transformando em um dataset treinável pelo Aprendizado Profundo

No código abaixo, altere o argumento "test_size" para que ele use 25% do dataset para teste.

In [5]:
raw_dataset = load_dataset('csv', data_files='data.csv')
raw_dataset = raw_dataset.class_encode_column('label')
split = raw_dataset['train'].train_test_split(test_size=0.25, seed=1993, stratify_by_column='label', shuffle=True)

Extracting data files:   0%|          | 0/1 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Casting to class labels:   0%|          | 0/2154 [00:00<?, ? examples/s]

### Baixando Modelo e Tokenizador

Tanto o modelo quanto o Tokenizador são modelos pré-treinados, ou seja, foram criados por outra pessoa. O que faremos aqui será um reajuste, adaptando e aproveitando um modelo que já conhece bem a língua portuguesa e alterando a última camada da sua rede neural.

Utilizaremos um modelo chamado BERT, desenvolvido originalmente pelo Google e que foi treinado posteriormente para a língua portuguesa.

Baixaremos o arquivo, colocando-o em um tokenizer e em um modelo, conforme explicado abaixo, e criaremos a função tokenizadora. A classe dataset existe que a função seja feita dessa forma, pois essa classe facilita o processamento: aqui, tanto a tokenização quanto o modelo processa as linhas em batch, ou blocos, e paraleliza usando a placa de vídeo. Assim, o processo é executado com mais rapidez.

Altere abaixo o número de labels existentes no dataset. O argumento é "num_labels" e ele está presente na função AutoModelForSequenceClassification.

In [6]:
checkpoint = 'neuralmind/bert-large-portuguese-cased'

tokenizer = AutoTokenizer.from_pretrained(checkpoint, model_max_length=1024, truncation=True)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels = 16)

def tokenize_fn(batch):
  return tokenizer(batch['text'], truncation=True)

tokenized_dataset = split.map(tokenize_fn, batched=True)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-large-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/1615 [00:00<?, ? examples/s]

Map:   0%|          | 0/539 [00:00<?, ? examples/s]

### Como o modelo é estruturado?

In [7]:
summary(model)

Layer (type:depth-idx)                                  Param #
BertForSequenceClassification                           --
├─BertModel: 1-1                                        --
│    └─BertEmbeddings: 2-1                              --
│    │    └─Embedding: 3-1                              30,509,056
│    │    └─Embedding: 3-2                              524,288
│    │    └─Embedding: 3-3                              2,048
│    │    └─LayerNorm: 3-4                              2,048
│    │    └─Dropout: 3-5                                --
│    └─BertEncoder: 2-2                                 --
│    │    └─ModuleList: 3-6                             302,309,376
│    └─BertPooler: 2-3                                  --
│    │    └─Linear: 3-7                                 1,049,600
│    │    └─Tanh: 3-8                                   --
├─Dropout: 1-2                                          --
├─Linear: 1-3                                           16,400
Total params

### Criando uma instância Trainer

O Trainer, dos transformers, é um facilitador: ele permite que treinemos um modelo com bastante facilidade, evitando preocupações com camadas, funções de ativação, etc. Precisamos apenas criar os argumentos (`training_args`), onde diremos por quantos epochs ele irá rodar; precisamos criar a métrica (`compute_metrics`) pra sabermos o quanto o modelo está acertando, e por fim, criamos a instância do Trainer (`trainer`), onde dizemos qual o modelo, os argumentos, o dataset de treino, o de avaliação (que seria o teste), qual o tokenizador e qual a função de computação de métricas. Altere o número de epochs abaixo para algo entre 3 e 10.

In [8]:
training_args = TrainingArguments(output_dir='modelo',
                                  evaluation_strategy='epoch',
                                  save_strategy='epoch',
                                  save_total_limit=3,
                                  save_safetensors=True,
                                  num_train_epochs=7,
                                  load_best_model_at_end=True,
                                  per_device_train_batch_size=4,
                                  per_device_eval_batch_size=16,
                                  )

def compute_metrics(logits_and_labels):
  logits, labels = logits_and_labels
  predictions = np.argmax(logits, axis=-1)
  acc = np.mean(predictions == labels)
  f1 = f1_score(labels, predictions, average='macro')
  return {'accuracy': acc, 'f1_score': f1}

trainer = Trainer(model,
                  training_args,
                  train_dataset=tokenized_dataset["train"],
                  eval_dataset=tokenized_dataset["test"],
                  tokenizer=tokenizer,
                  compute_metrics=compute_metrics)

### Treinando...

In [9]:
trainer.train()

You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Score
1,No log,0.915855,0.781076,0.699424
2,1.112300,1.020217,0.784787,0.703147
3,0.580800,0.972551,0.827458,0.793307
4,0.324700,1.172975,0.801484,0.755728
5,0.172000,1.134950,0.812616,0.777222
6,0.172000,1.263134,0.808905,0.778292
7,0.057600,1.254618,0.807050,0.775588


TrainOutput(global_step=2828, training_loss=0.40145774137181306, metrics={'train_runtime': 1102.7183, 'train_samples_per_second': 10.252, 'train_steps_per_second': 2.565, 'total_flos': 1444396298044896.0, 'train_loss': 0.40145774137181306, 'epoch': 7.0})

### Testando!

Agora vamos testar o modelo!

In [11]:
from transformers import pipeline

dicionario = {0: 'Ameaça Ao Estado Democrático De Direito', 1: 'Calendários Oficiais, Patrimonialização E Homenagens',
              2: 'Causa Animal', 3: 'Crimes E Segurança Pública', 4: 'Direitos Da Criança E Adolescente',
              5: 'Direitos Da Mulher', 6: 'Direitos Dos Povos Indígenas E Comunidades Tradicionais', 7: 'Direitos Humanos',
              8: 'Direitos Sexuais E Reprodutivos', 9: 'Educação', 10: 'Família', 11: 'Imigração',
              12: 'Políticas De Drogas', 13: 'Raça E Racismo', 14: 'Regulação Das Mídias', 15: 'Religião'}

classificador = pipeline('text-classification', model='/content/modelo/checkpoint-2828', tokenizer='/content/modelo/checkpoint-2828')

classificador.model.config.id2label = dicionario

In [12]:
classificador('Dispõe sobre as diretrizes para combate a noticias falsas na internet.')

[{'label': 'Regulação Das Mídias', 'score': 0.9996614456176758}]